# Part I — The Mathematical Engine

**Exercises from *Machine Learning: The Working Text*.**

Norms, Gram–Schmidt, LU, conditioning, eigenpairs, the SVD, PCA, gradient descent and its stability limit, Jacobians, backpropagation, entropy, KL divergence and maximum likelihood.

---

## How to use this notebook

1. Run the **Setup** cell below once.
2. Work down the notebook. Each exercise has a stub cell — replace the
   `raise NotImplementedYet(...)` with your own code and run it.
3. Run the `check(...)` cell under it. It prints `PASS`, `FAIL` or `TODO`.
4. `check_all()` at the bottom summarises the notebook.

**A `TODO` is never a pass.** The gate thresholds come from the book, so
when one fails the useful question is which of the two is wrong — your
code, or your understanding of the threshold.

Nothing needs installing: Colab already has NumPy and PyTorch. Nothing
is downloaded, and no file is read from disk.

### Exercises in this notebook

| id | exercise | read first |
|---|---|---|
| `1.1` | l_p norms match NumPy | Ch 2 · MML p.49 |
| `1.2` | Vectorisation speedup | Ch 1 · MML p.37 |
| `1.3` | Rank detects collinearity | Ch 1 · MML p.15 |
| `2.1` | Gram-Schmidt orthonormality | Ch 2 · MML p.89 |
| `2.2` | Classical vs modified on a Hilbert matrix | Ch 2 · MML p.67 |
| `2.3` | Projection is idempotent and symmetric | Ch 2 |
| `3.1` | Rank-nullity theorem | Ch 3 · MML p.127 |
| `3.2` | Rotations preserve length | Ch 3 · MML p.134 |
| `4.1` | LU decomposition: PA = LU | Ch 4 · MML p.171 |
| `4.2` | Solving from the factorisation | Ch 4 |
| `4.3` | Determinant via LU | Ch 4 · MML p.182 |
| `4.4` | Solve beats explicit inversion | Ch 4 · MML p.177 |
| `5.1` | Power iteration | Ch 5 · MML p.217 |
| `5.2` | Eckart-Young low-rank error | Ch 5 · MML p.207 |
| `5.3` | PCA from the SVD | Ch 5 |
| `5.4` | Eigenvalues of a 2x2 by hand | Ch 5 · MML p.192 |
| `5.5` | Singular values from AtA | Ch 5 · MML p.207 |
| `6.1` | The U-shaped differentiation error | Ch 6 |
| `6.2` | Gradient descent converges below 2/L | Ch 6 · MML p.381 |
| `6.3` | Gradient descent diverges above 2/L | Ch 6 · MML p.389 |
| `7.1` | Numeric Jacobian | Ch 7 · MML p.460 |
| `7.2` | Backpropagation by hand | Ch 7 |
| `7.3` | Curse of dimensionality | Ch 7 · MML p.443 |
| `8.1` | Entropy | Ch 8 · MML p.606 |
| `8.2` | KL divergence properties | Ch 8 · MML p.606 |
| `8.3` | Maximum likelihood recovers parameters | Ch 8 · MML p.619 |
| `8.4` | Welford beats the naive variance | Ch 8 · MML p.589 |
| `8.5` | Bayes and the base rate | Ch 8 · MML p.515 |

In [ ]:
#@title Setup — run this cell first (collapsed: it holds the gates and the reference solutions)

import math
import time
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import torch
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False


class NotImplementedYet(NotImplementedError):
    """Raised by a stub you have not filled in yet."""


def _rng(seed: int = 0) -> np.random.Generator:
    """Returns a seeded generator so every gate is reproducible."""
    return np.random.default_rng(seed)


def _fit_predict(x_train, y_train, x_test):
    """Least-squares probe used by the leakage gates."""
    design = np.hstack([x_train, np.ones((x_train.shape[0], 1))])
    weights, *_ = np.linalg.lstsq(design, y_train, rcond=None)
    return np.hstack([x_test, np.ones((x_test.shape[0], 1))]) @ weights


def _r2(y_true, y_pred):
    """Coefficient of determination."""
    total = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - float(np.sum((y_true - y_pred) ** 2)) / total


# Reference implementations. Two gates compare against these, so
# they have to be here. Reading them before you have tried is a
# way of not learning the material.
class Solutions:
    """Reference implementations of every exercise."""

    @staticmethod
    def lp_norm(x: np.ndarray, p: float) -> float:
        """Returns the l_p norm of a vector."""
        x = np.asarray(x, dtype=np.float64)
        if np.isinf(p):
            return float(np.max(np.abs(x)))
        return float(np.sum(np.abs(x) ** p) ** (1.0 / p))

    @staticmethod
    def gram_schmidt_classical(a: np.ndarray) -> np.ndarray:
        """Orthonormalises columns, projecting against the original column."""
        a = np.asarray(a, dtype=np.float64)
        n, k = a.shape
        q = np.zeros((n, k))
        for j in range(k):
            v = a[:, j].copy()
            for i in range(j):
                v = v - (q[:, i] @ a[:, j]) * q[:, i]
            norm = np.linalg.norm(v)
            if norm < 1e-300:
                raise ValueError(f'column {j} is dependent')
            q[:, j] = v / norm
        return q

    @staticmethod
    def gram_schmidt_modified(a: np.ndarray) -> np.ndarray:
        """Orthonormalises columns, projecting against the running residual."""
        a = np.asarray(a, dtype=np.float64)
        n, k = a.shape
        q = np.zeros((n, k))
        for j in range(k):
            v = a[:, j].copy()
            for i in range(j):
                v = v - (q[:, i] @ v) * q[:, i]
            norm = np.linalg.norm(v)
            if norm < 1e-300:
                raise ValueError(f'column {j} is dependent')
            q[:, j] = v / norm
        return q

    @staticmethod
    def projection_matrix(v: np.ndarray) -> np.ndarray:
        """Returns the matrix projecting onto span(v)."""
        v = np.asarray(v, dtype=np.float64).reshape(-1, 1)
        return (v @ v.T) / float((v.T @ v).item())

    @staticmethod
    def lu_decompose(a: np.ndarray) -> Tuple[np.ndarray, np.ndarray,
                                             np.ndarray]:
        """Factors a square matrix as PA = LU with partial pivoting."""
        a = np.asarray(a, dtype=np.float64)
        n = a.shape[0]
        u = a.copy()
        l = np.eye(n)
        p = np.eye(n)
        for k in range(n - 1):
            pivot = int(np.argmax(np.abs(u[k:, k]))) + k
            if abs(u[pivot, k]) < 1e-14:
                raise ValueError('singular to working precision')
            if pivot != k:
                u[[k, pivot], k:] = u[[pivot, k], k:]
                l[[k, pivot], :k] = l[[pivot, k], :k]
                p[[k, pivot], :] = p[[pivot, k], :]
            for row in range(k + 1, n):
                factor = u[row, k] / u[k, k]
                l[row, k] = factor
                u[row, k:] -= factor * u[k, k:]
        return p, l, u

    @staticmethod
    def lu_solve(p: np.ndarray, l: np.ndarray, u: np.ndarray,
                 b: np.ndarray) -> np.ndarray:
        """Solves Ax = b given PA = LU."""
        pb = p @ np.asarray(b, dtype=np.float64)
        n = len(pb)
        y = np.zeros(n)
        for i in range(n):
            y[i] = pb[i] - l[i, :i] @ y[:i]
        x = np.zeros(n)
        for i in range(n - 1, -1, -1):
            x[i] = (y[i] - u[i, i + 1:] @ x[i + 1:]) / u[i, i]
        return x

    @staticmethod
    def det_via_lu(a: np.ndarray) -> float:
        """Returns det(A) from its LU factorisation."""
        p, _, u = Solutions.lu_decompose(a)
        swaps = int(round(np.sum(np.abs(p - np.eye(p.shape[0]))) / 2.0))
        return float(((-1.0) ** swaps) * np.prod(np.diag(u)))

    @staticmethod
    def power_iteration(a: np.ndarray) -> Tuple[float, np.ndarray]:
        """Returns the dominant eigenpair of a symmetric matrix."""
        a = np.asarray(a, dtype=np.float64)
        rng = np.random.default_rng(0)
        v = rng.normal(size=a.shape[0])
        v /= np.linalg.norm(v)
        for _ in range(10_000):
            w = a @ v
            norm = np.linalg.norm(w)
            if norm < 1e-300:
                return 0.0, v
            w /= norm
            if min(np.linalg.norm(w - v), np.linalg.norm(w + v)) < 1e-14:
                v = w
                break
            v = w
        return float(v @ a @ v), v

    @staticmethod
    def svd_lowrank(a: np.ndarray, k: int) -> np.ndarray:
        """Returns the best rank-k approximation."""
        u, s, vt = np.linalg.svd(np.asarray(a, dtype=np.float64),
                                 full_matrices=False)
        return (u[:, :k] * s[:k]) @ vt[:k, :]

    @staticmethod
    def pca(x: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
        """Returns (components, explained_variance_ratio)."""
        x = np.asarray(x, dtype=np.float64)
        centred = x - x.mean(axis=0)
        _, s, vt = np.linalg.svd(centred, full_matrices=False)
        variance = s ** 2 / (x.shape[0] - 1)
        return vt[:k], variance[:k] / variance.sum()

    @staticmethod
    def central_difference(f: Callable, x: float, h: float) -> float:
        """Approximates f'(x) by central difference."""
        return float((f(x + h) - f(x - h)) / (2.0 * h))

    @staticmethod
    def gradient_descent(grad: Callable, x0: np.ndarray, lr: float,
                         iters: int, momentum: float = 0.0) -> np.ndarray:
        """Minimises a function given its gradient."""
        x = np.asarray(x0, dtype=np.float64).copy()
        velocity = np.zeros_like(x)
        for _ in range(iters):
            g = np.asarray(grad(x), dtype=np.float64)
            velocity = momentum * velocity - lr * g
            x = x + velocity
            if not np.all(np.isfinite(x)):
                break
        return x

    @staticmethod
    def numeric_jacobian(f: Callable, x: np.ndarray,
                         h: float = 1e-6) -> np.ndarray:
        """Approximates the Jacobian of f at x."""
        x = np.asarray(x, dtype=np.float64)
        base = np.atleast_1d(np.asarray(f(x), dtype=np.float64))
        jac = np.zeros((base.size, x.size))
        for i in range(x.size):
            step = np.zeros_like(x)
            step[i] = h
            plus = np.atleast_1d(np.asarray(f(x + step), dtype=np.float64))
            minus = np.atleast_1d(np.asarray(f(x - step), dtype=np.float64))
            jac[:, i] = (plus - minus) / (2.0 * h)
        return jac

    @staticmethod
    def two_layer_backward(w1: np.ndarray, b1: np.ndarray, w2: np.ndarray,
                           b2: np.ndarray, x: np.ndarray,
                           y: float) -> Tuple[float, Dict[str, np.ndarray]]:
        """Forward and backward pass of a two-layer tanh network."""
        z1 = w1 @ x + b1
        a1 = np.tanh(z1)
        z2 = w2 @ a1 + b2
        loss = 0.5 * float(((z2 - y) ** 2).item())
        d2 = (z2 - y)
        grads = {'w2': np.outer(d2, a1), 'b2': d2}
        d1 = (w2.T @ d2) * (1.0 - a1 ** 2)
        grads['w1'] = np.outer(d1, x)
        grads['b1'] = d1
        return loss, grads

    @staticmethod
    def shell_fraction(d: int, eps: float) -> float:
        """Fraction of a unit ball's volume within eps of the surface."""
        return float(1.0 - (1.0 - eps) ** d)

    @staticmethod
    def entropy(p: np.ndarray) -> float:
        """Shannon entropy in nats."""
        p = np.asarray(p, dtype=np.float64)
        nonzero = p[p > 0]
        return float(-np.sum(nonzero * np.log(nonzero)))

    @staticmethod
    def kl_divergence(p: np.ndarray, q: np.ndarray) -> float:
        """KL divergence from q to p, in nats."""
        p = np.asarray(p, dtype=np.float64)
        q = np.asarray(q, dtype=np.float64)
        mask = p > 0
        if np.any(q[mask] <= 0):
            raise ValueError('KL is infinite where q = 0 and p > 0')
        return float(np.sum(p[mask] * np.log(p[mask] / q[mask])))

    @staticmethod
    def mle_gaussian(samples: np.ndarray) -> Tuple[float, float]:
        """Maximum-likelihood mean and variance."""
        x = np.asarray(samples, dtype=np.float64)
        mu = float(x.mean())
        return mu, float(((x - mu) ** 2).mean())

    @staticmethod
    def welford(values: np.ndarray) -> Tuple[float, float]:
        """One-pass stable mean and sample variance."""
        count, mean, m2 = 0, 0.0, 0.0
        for value in np.asarray(values, dtype=np.float64).ravel():
            count += 1
            delta = value - mean
            mean += delta / count
            m2 += delta * (value - mean)
        return mean, (m2 / (count - 1) if count > 1 else 0.0)

    @staticmethod
    def bayes_posterior(sensitivity: float, specificity: float,
                        prevalence: float) -> float:
        """P(disease | positive test)."""
        true_pos = sensitivity * prevalence
        false_pos = (1.0 - specificity) * (1.0 - prevalence)
        return float(true_pos / (true_pos + false_pos))

    @staticmethod
    def select_features(x_train: np.ndarray, y_train: np.ndarray,
                        k: int) -> np.ndarray:
        """Indices of the k training-correlated features."""
        centred = y_train - y_train.mean()
        scores = np.abs(x_train.T @ centred)
        return np.argsort(scores)[-k:]

    @staticmethod
    def assert_no_lookforward(folds) -> None:
        """Raises AssertionError if a fold leaks the future."""
        for i, (train_idx, test_idx) in enumerate(folds):
            if len(train_idx) == 0 or len(test_idx) == 0:
                raise AssertionError(f'fold {i}: empty split')
            if int(np.max(train_idx)) >= int(np.min(test_idx)):
                raise AssertionError(f'fold {i}: lookforward leak')

    @staticmethod
    def standardize_fit(x_train: np.ndarray) -> Tuple[np.ndarray,
                                                      np.ndarray]:
        """Returns (mean, std) from the training rows only."""
        x_train = np.asarray(x_train, dtype=np.float64)
        std = x_train.std(axis=0)
        std[std == 0] = 1.0
        return x_train.mean(axis=0), std

    @staticmethod
    def ridge_fit(x: np.ndarray, y: np.ndarray, alpha: float) -> np.ndarray:
        """Ridge coefficients with an unpenalised intercept, placed last."""
        if alpha < 0:
            raise ValueError('alpha must be non-negative')
        x = np.asarray(x, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        x_mean, y_mean = x.mean(axis=0), y.mean()
        xc, yc = x - x_mean, y - y_mean
        d = x.shape[1]
        w = np.linalg.solve(xc.T @ xc + alpha * np.eye(d), xc.T @ yc)
        return np.append(w, y_mean - x_mean @ w)

    @staticmethod
    def logistic_gradient(w: np.ndarray, x: np.ndarray,
                          y: np.ndarray) -> np.ndarray:
        """Gradient of mean binary cross-entropy with respect to w."""
        z = x @ w
        p = 1.0 / (1.0 + np.exp(-z))
        return x.T @ (p - y) / len(y)

    @staticmethod
    def gini_impurity(labels: np.ndarray) -> float:
        """Gini impurity of a label vector."""
        labels = np.asarray(labels)
        if labels.size == 0:
            return 0.0
        _, counts = np.unique(labels, return_counts=True)
        p = counts / labels.size
        return float(1.0 - np.sum(p ** 2))

    @staticmethod
    def precision_recall_f1(y_true: np.ndarray,
                            y_pred: np.ndarray) -> Tuple[float, float,
                                                         float]:
        """Binary precision, recall and F1 for the positive class."""
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        tp = int(np.sum((y_true == 1) & (y_pred == 1)))
        fp = int(np.sum((y_true == 0) & (y_pred == 1)))
        fn = int(np.sum((y_true == 1) & (y_pred == 0)))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        denom = precision + recall
        f1 = 2 * precision * recall / denom if denom else 0.0
        return precision, recall, f1

    @staticmethod
    def bootstrap_ci(metric_fn: Callable, y_true: np.ndarray,
                     y_pred: np.ndarray, n_boot: int, level: float,
                     seed: int) -> Tuple[float, float]:
        """Percentile bootstrap confidence interval."""
        rng = np.random.default_rng(seed)
        n = len(y_true)
        samples = np.empty(n_boot)
        for i in range(n_boot):
            idx = rng.integers(0, n, size=n)
            samples[i] = metric_fn(y_true[idx], y_pred[idx])
        tail = (1.0 - level) / 2.0
        return (float(np.quantile(samples, tail)),
                float(np.quantile(samples, 1.0 - tail)))

    @staticmethod
    def kmeans(x: np.ndarray, k: int, iters: int,
               seed: int) -> Tuple[np.ndarray, np.ndarray, List[float]]:
        """Lloyd's algorithm with a fixed seed."""
        x = np.asarray(x, dtype=np.float64)
        rng = np.random.default_rng(seed)
        centroids = x[rng.choice(len(x), size=k, replace=False)].copy()
        history: List[float] = []
        labels = np.zeros(len(x), dtype=int)
        for _ in range(iters):
            distances = ((x[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
            labels = np.argmin(distances, axis=1)
            history.append(float(distances[np.arange(len(x)), labels].sum()))
            for j in range(k):
                members = x[labels == j]
                if len(members):
                    centroids[j] = members.mean(axis=0)
        return labels, centroids, history

    @staticmethod
    def stable_softmax(z: np.ndarray) -> np.ndarray:
        """Softmax over the last axis, overflow-safe."""
        z = np.asarray(z, dtype=np.float64)
        shifted = z - z.max(axis=-1, keepdims=True)
        exp = np.exp(shifted)
        return exp / exp.sum(axis=-1, keepdims=True)

    @staticmethod
    def cross_entropy_grad(logits: np.ndarray,
                           one_hot: np.ndarray) -> np.ndarray:
        """Gradient of cross-entropy with respect to the logits."""
        return Solutions.stable_softmax(logits) - one_hot

    @staticmethod
    def conv2d(x: np.ndarray, kernel: np.ndarray) -> np.ndarray:
        """Valid single-channel 2-D cross-correlation."""
        x = np.asarray(x, dtype=np.float64)
        kernel = np.asarray(kernel, dtype=np.float64)
        kh, kw = kernel.shape
        out_h, out_w = x.shape[0] - kh + 1, x.shape[1] - kw + 1
        patches = np.lib.stride_tricks.sliding_window_view(x, (kh, kw))
        cols = patches.reshape(out_h * out_w, kh * kw)
        return (cols @ kernel.ravel()).reshape(out_h, out_w)

    @staticmethod
    def attention(q: np.ndarray, k: np.ndarray,
                  v: np.ndarray) -> np.ndarray:
        """Scaled dot-product attention."""
        scores = (q @ k.T) / np.sqrt(k.shape[-1])
        return Solutions.stable_softmax(scores) @ v

    @staticmethod
    def torch_training_step(model, xb, yb, optimiser, loss_fn) -> float:
        """One zero_grad / forward / backward / step."""
        optimiser.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimiser.step()
        # .detach() first: converting a tensor that still carries
        # requires_grad straight to a float warns, and in a loop it
        # quietly keeps the graph alive.
        return float(loss.detach())

    @staticmethod
    def acf(series: np.ndarray, max_lag: int) -> np.ndarray:
        """Sample autocorrelation for lags 0..max_lag."""
        y = np.asarray(series, dtype=np.float64)
        y = y - y.mean()
        denom = float(y @ y)
        return np.array([float(y[k:] @ y[:len(y) - k]) / denom
                         for k in range(max_lag + 1)])

    @staticmethod
    def additive_decompose(series: np.ndarray,
                           period: int) -> Tuple[np.ndarray, np.ndarray,
                                                 np.ndarray]:
        """Centred-moving-average additive decomposition."""
        y = np.asarray(series, dtype=np.float64)
        kernel = np.ones(period) / period
        trend = np.convolve(y, kernel, mode='same')
        half = period // 2
        trend[:half] = trend[half]
        trend[len(y) - half:] = trend[len(y) - half - 1]
        detrended = y - trend
        seasonal_means = np.array([detrended[i::period].mean()
                                   for i in range(period)])
        seasonal_means -= seasonal_means.mean()
        seasonal = np.array([seasonal_means[i % period]
                             for i in range(len(y))])
        return trend, seasonal, y - trend - seasonal

    @staticmethod
    def rolling_window_cv(n: int, n_folds: int, horizon: int,
                          window: int) -> List[Tuple[np.ndarray,
                                                     np.ndarray]]:
        """Rolling-origin folds with a constant training length."""
        needed = window + n_folds * horizon
        if needed > n:
            raise ValueError(f'need {needed} points, have {n}')
        folds = []
        for i in range(n_folds):
            end = window + i * horizon
            folds.append((np.arange(end - window, end),
                          np.arange(end, end + horizon)))
        return folds

    @staticmethod
    def naive_scale(y_train: np.ndarray, seasonality: int) -> float:
        """MAE of the seasonal naive forecast on the training series."""
        y = np.asarray(y_train, dtype=np.float64)
        if y.size <= seasonality:
            raise ValueError('training series too short')
        scale = float(np.mean(np.abs(y[seasonality:] - y[:-seasonality])))
        if scale == 0.0:
            raise ValueError('constant training series; MASE undefined')
        return scale

    @staticmethod
    def mase(y_true: np.ndarray, y_pred: np.ndarray, y_train: np.ndarray,
             seasonality: int) -> float:
        """Mean absolute scaled error."""
        mae = float(np.mean(np.abs(np.asarray(y_true, dtype=np.float64) -
                                   np.asarray(y_pred, dtype=np.float64))))
        return mae / Solutions.naive_scale(y_train, seasonality)

    @staticmethod
    def absolute_percentage_error(y_true: np.ndarray,
                                  y_pred: np.ndarray) -> np.ndarray:
        """Element-wise APE, undefined at zero targets."""
        y_true = np.asarray(y_true, dtype=np.float64)
        y_pred = np.asarray(y_pred, dtype=np.float64)
        if np.any(y_true == 0.0):
            raise ValueError('APE is undefined for zero targets [TSP p.144]')
        return np.abs((y_true - y_pred) / y_true)

    @staticmethod
    def split_conformal(residuals: np.ndarray, alpha: float) -> float:
        """Conformal half-width for 1 - alpha coverage."""
        s = np.sort(np.asarray(residuals, dtype=np.float64))
        n = s.size
        k = int(np.ceil((n + 1) * (1.0 - alpha))) - 1
        return float(s[min(max(k, 0), n - 1)])

    @staticmethod
    def eigenvalues_2x2(a: np.ndarray) -> Tuple[float, float]:
        """Eigenvalues of a 2x2 from the characteristic polynomial."""
        a = np.asarray(a, dtype=np.float64)
        trace = float(a[0, 0] + a[1, 1])
        det = float(a[0, 0] * a[1, 1] - a[0, 1] * a[1, 0])
        disc = trace ** 2 - 4.0 * det
        if disc < 0:
            raise ValueError('complex eigenvalues; this exercise is real-only')
        root = float(np.sqrt(disc))
        return (trace + root) / 2.0, (trace - root) / 2.0

    @staticmethod
    def singular_values(a: np.ndarray) -> np.ndarray:
        """Singular values as the square roots of the eigenvalues of AᵀA."""
        a = np.asarray(a, dtype=np.float64)
        eigenvalues = np.linalg.eigvalsh(a.T @ a)
        clipped = np.clip(eigenvalues, 0.0, None)
        return np.sqrt(clipped)[::-1]

    @staticmethod
    def optimal_threshold(cost_fp: float, cost_fn: float) -> float:
        """Decision threshold balancing the two expected costs."""
        if cost_fp < 0 or cost_fn < 0 or (cost_fp + cost_fn) == 0:
            raise ValueError('costs must be non-negative and not both zero')
        return float(cost_fp / (cost_fp + cost_fn))

    @staticmethod
    def soft_threshold(c: float, alpha: float) -> float:
        """The 1-D lasso solution."""
        return float(np.sign(c) * max(abs(c) - alpha, 0.0))

    @staticmethod
    def gini_gain(parent: np.ndarray, left: np.ndarray,
                  right: np.ndarray) -> float:
        """Weighted Gini impurity reduction of a split."""
        n = len(parent)
        if n == 0 or len(left) + len(right) != n:
            raise ValueError('children must partition the parent')
        weighted = (len(left) / n) * Solutions.gini_impurity(left) + \
                   (len(right) / n) * Solutions.gini_impurity(right)
        return float(Solutions.gini_impurity(parent) - weighted)

    @staticmethod
    def conv_output_size(in_size: int, kernel: int, stride: int,
                         padding: int, dilation: int) -> int:
        """Convolution output size along one axis."""
        effective = dilation * (kernel - 1)
        return int((in_size + 2 * padding - effective - 1) // stride) + 1

    @staticmethod
    def ar1_acf(phi: float, max_lag: int) -> np.ndarray:
        """Theoretical ACF of an AR(1)."""
        if abs(phi) >= 1.0:
            raise ValueError('|phi| must be < 1 for a stationary AR(1)')
        return np.array([phi ** k for k in range(max_lag + 1)])

    @staticmethod
    def winkler_score(y: float, lower: float, upper: float,
                      alpha: float) -> float:
        """Interval score: width plus an out-of-interval penalty."""
        width = upper - lower
        if y < lower:
            return float(width + (2.0 / alpha) * (lower - y))
        if y > upper:
            return float(width + (2.0 / alpha) * (y - upper))
        return float(width)


# ---- gates ----------------------------------------------------
def _g_norm(impl) -> str:
    x = _rng(0).normal(size=64)
    for p in (1.0, 2.0, float('inf')):
        got = impl.lp_norm(x, p)
        want = float(np.linalg.norm(x, ord=p))
        assert abs(got - want) < 1e-10 * max(1.0, abs(want)), f'p={p}'
    assert abs(impl.lp_norm(np.array([3.0, 4.0]), 2.0) - 5.0) < 1e-12
    return 'matches numpy for p = 1, 2, inf'


def _g_vectorisation(impl) -> str:
    n = 200_000
    a, b = _rng(1).normal(size=n), _rng(2).normal(size=n)
    start = time.perf_counter()
    slow = sum(x * y for x, y in zip(a.tolist(), b.tolist()))
    py_s = time.perf_counter() - start
    start = time.perf_counter()
    fast = float(np.dot(a, b))
    np_s = max(time.perf_counter() - start, 1e-9)
    assert abs(slow - fast) < 1e-6 * max(1.0, abs(fast))
    return f'speedup on this machine: {py_s / np_s:.0f}x'


def _g_rank(impl) -> str:
    x = _rng(3).normal(size=(50, 5))
    x[:, 4] = x[:, 0] + 2.0 * x[:, 1]          # deliberately collinear
    rank = int(np.linalg.matrix_rank(x))
    assert rank == 4, f'expected rank 4, got {rank}'
    return 'collinear column detected (rank 4 of 5)'


def _g_gram_schmidt(impl) -> str:
    a = _rng(4).normal(size=(50, 6))
    q = impl.gram_schmidt_modified(a)
    err = float(np.abs(q.T @ q - np.eye(6)).max())
    assert err < 1e-10, f'QtQ deviates by {err:.2e}'
    return f'QtQ = I to {err:.1e}'


def _g_gram_schmidt_stability(impl) -> str:
    n = 10
    hilbert = np.array([[1.0 / (i + j + 1) for j in range(n)]
                        for i in range(n)])
    classical = float(np.abs(impl.gram_schmidt_classical(hilbert).T @
                             impl.gram_schmidt_classical(hilbert) -
                             np.eye(n)).max())
    modified = float(np.abs(impl.gram_schmidt_modified(hilbert).T @
                            impl.gram_schmidt_modified(hilbert) -
                            np.eye(n)).max())
    assert modified < classical, (
        f'modified ({modified:.2e}) should beat classical ({classical:.2e})')
    return f'classical {classical:.1e} vs modified {modified:.1e}'


def _g_projection(impl) -> str:
    v = _rng(5).normal(size=7)
    p = impl.projection_matrix(v)
    assert np.allclose(p @ p, p, atol=1e-12), 'P² != P'
    assert np.allclose(p, p.T, atol=1e-12), 'P != Pᵀ'
    return 'idempotent and symmetric to 1e-12'


def _g_rank_nullity(impl) -> str:
    rng = _rng(6)
    for _ in range(50):
        a = rng.normal(size=(8, 6)) @ rng.normal(size=(6, 6))
        rank = int(np.linalg.matrix_rank(a))
        nullity = a.shape[1] - rank
        assert rank + nullity == a.shape[1]
    return 'rank + nullity = n over 50 matrices'


def _g_rotation(impl) -> str:
    theta = 0.7
    rot = np.array([[np.cos(theta), -np.sin(theta)],
                    [np.sin(theta), np.cos(theta)]])
    x = _rng(7).normal(size=2)
    assert abs(np.linalg.norm(rot @ x) - np.linalg.norm(x)) < 1e-12
    assert abs(np.linalg.det(rot) - 1.0) < 1e-12
    return 'norm preserved, determinant 1'


def _g_lu(impl) -> str:
    a = _rng(8).normal(size=(8, 8))
    p, l, u = impl.lu_decompose(a)
    err = float(np.abs(p @ a - l @ u).max())
    assert err < 1e-10, f'||PA - LU|| = {err:.2e}'
    assert np.allclose(np.tril(l), l, atol=1e-12), 'L is not lower-triangular'
    assert np.allclose(np.triu(u), u, atol=1e-12), 'U is not upper-triangular'
    return f'PA = LU to {err:.1e}'


def _g_lu_solve(impl) -> str:
    rng = _rng(9)
    a = rng.normal(size=(10, 10))
    b = rng.normal(size=10)
    p, l, u = impl.lu_decompose(a)
    mine = impl.lu_solve(p, l, u, b)
    want = np.linalg.solve(a, b)
    err = float(np.abs(mine - want).max())
    assert err < 1e-9, f'solution differs by {err:.2e}'
    return f'matches np.linalg.solve to {err:.1e}'


def _g_det(impl) -> str:
    a = _rng(10).normal(size=(7, 7))
    mine, want = impl.det_via_lu(a), float(np.linalg.det(a))
    err = abs(mine - want) / max(1.0, abs(want))
    assert err < 1e-9, f'relative error {err:.2e}'
    return f'matches np.linalg.det to {err:.1e} relative'


def _g_conditioning(impl) -> str:
    rng = _rng(11)
    n = 120
    u_mat, _, vt = np.linalg.svd(rng.normal(size=(n, n)))
    a = (u_mat * np.logspace(0, -10, n)) @ vt
    x_true = rng.normal(size=n)
    b = a @ x_true
    err_inv = float(np.linalg.norm(np.linalg.inv(a) @ b - x_true))
    err_solve = float(np.linalg.norm(np.linalg.solve(a, b) - x_true))
    assert err_solve <= err_inv, (
        f'solve {err_solve:.2e} should not exceed inv {err_inv:.2e}')
    return f'kappa {np.linalg.cond(a):.1e}; solve {err_solve:.1e} <= inv {err_inv:.1e}'


def _g_power_iteration(impl) -> str:
    base = _rng(12).normal(size=(20, 20))
    a = base @ base.T
    value, vector = impl.power_iteration(a)
    want = float(np.linalg.eigvalsh(a).max())
    assert abs(value - want) < 1e-6 * want, f'{value:.6f} vs {want:.6f}'
    residual = float(np.linalg.norm(a @ vector - value * vector))
    assert residual < 1e-6 * want, f'residual {residual:.2e}'
    return f'dominant eigenvalue to {abs(value - want) / want:.1e} relative'


def _g_eckart_young(impl) -> str:
    a = _rng(13).normal(size=(40, 15))
    s = np.linalg.svd(a, compute_uv=False)
    details = []
    for k in (1, 5, 10):
        predicted = float(np.sqrt(np.sum(s[k:] ** 2)))
        actual = float(np.linalg.norm(a - impl.svd_lowrank(a, k)))
        assert abs(predicted - actual) < 1e-9, f'k={k}'
        details.append(f'k={k}:{actual:.3f}')
    return 'Eckart-Young exact at ' + ', '.join(details)


def _g_pca(impl) -> str:
    rng = _rng(14)
    latent = rng.normal(size=(300, 2))
    mixing = rng.normal(size=(2, 5))
    x = latent @ mixing + 0.05 * rng.normal(size=(300, 5))
    components, ratio = impl.pca(x, 2)
    centred = x - x.mean(axis=0)
    cov = centred.T @ centred / (len(x) - 1)
    eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
    want = eigvals[:2] / eigvals.sum()
    assert np.allclose(ratio, want, atol=1e-8), 'variance ratios differ'
    assert abs(float(np.abs(components @ components.T)[0, 1])) < 1e-8
    return f'top-2 variance {ratio.sum():.3f}, components orthogonal'


def _g_eig_2x2(impl) -> str:
    a = np.array([[4.0, 1.0], [2.0, 3.0]])
    high, low = impl.eigenvalues_2x2(a)
    assert abs(high - 5.0) < 1e-12 and abs(low - 2.0) < 1e-12, (high, low)
    assert abs((high + low) - np.trace(a)) < 1e-12, 'trace must equal the sum'
    assert abs(high * low - np.linalg.det(a)) < 1e-12, 'det must equal product'
    return 'eigenvalues 5 and 2; trace and determinant identities hold'


def _g_singular_values(impl) -> str:
    a = np.array([[3.0, 0.0], [4.0, 5.0]])
    got = np.sort(np.asarray(impl.singular_values(a)))[::-1]
    want = np.sort(np.linalg.svd(a, compute_uv=False))[::-1]
    assert np.allclose(got, want, atol=1e-10), f'{got} vs {want}'
    assert abs(float(np.prod(got)) - abs(float(np.linalg.det(a)))) < 1e-10, (
        'the product of singular values must equal |det|')
    return f'sigma = ({got[0]:.4f}, {got[1]:.4f}); product equals |det| = 15'


def _g_central_difference(impl) -> str:
    exact = float(np.cos(1.0))
    errors = [abs(impl.central_difference(np.sin, 1.0, 10.0 ** -k) - exact)
              for k in range(1, 15)]
    best = int(np.argmin(errors))
    assert best not in (0, len(errors) - 1), (
        'the error curve should bottom out in the middle, not at an end')
    assert errors[best] < 1e-9
    return f'minimum error {errors[best]:.1e} at h = 1e-{best + 1}'


def _g_gd_converges(impl) -> str:
    curvature = np.array([4.0, 1.0])
    limit = 2.0 / (2.0 * curvature.max())
    grad = lambda x: 2.0 * curvature * x
    final = impl.gradient_descent(grad, np.array([5.0, -3.0]),
                                  0.5 * limit, 2000)
    size = float(np.linalg.norm(final))
    assert size < 1e-6, f'did not converge: ||x|| = {size:.2e}'
    return f'||x|| = {size:.1e} at h = 0.5 x limit'


def _g_gd_diverges(impl) -> str:
    curvature = np.array([4.0, 1.0])
    limit = 2.0 / (2.0 * curvature.max())
    grad = lambda x: 2.0 * curvature * x
    final = impl.gradient_descent(grad, np.array([5.0, -3.0]),
                                  1.5 * limit, 200)
    size = float(np.linalg.norm(final))
    assert (not np.isfinite(size)) or size > 1e3, (
        'gradient descent must diverge above 2/L; yours did not')
    return 'diverges above 2/L, as required'


def _g_jacobian(impl) -> str:
    f = lambda x: np.array([x[0] ** 2 + x[1], np.sin(x[2]) * x[0]])
    analytic = lambda x: np.array([[2 * x[0], 1.0, 0.0],
                                   [np.sin(x[2]), 0.0, np.cos(x[2]) * x[0]]])
    x = np.array([0.7, -1.3, 0.4])
    err = float(np.abs(impl.numeric_jacobian(f, x) - analytic(x)).max())
    assert err < 1e-6, f'max gap {err:.2e}'
    return f'matches analytic to {err:.1e}'


def _g_backprop(impl) -> str:
    rng = _rng(15)
    w1, b1 = rng.normal(size=(4, 3)), rng.normal(size=4)
    w2, b2 = rng.normal(size=(1, 4)), rng.normal(size=1)
    x, y = rng.normal(size=3), 0.4
    _, grads = impl.two_layer_backward(w1, b1, w2, b2, x, y)

    def loss_of(w1_, b1_, w2_, b2_):
        a1 = np.tanh(w1_ @ x + b1_)
        return 0.5 * float(((w2_ @ a1 + b2_ - y) ** 2).item())

    h = 1e-6
    numeric = np.zeros_like(w1)
    for i in range(w1.shape[0]):
        for j in range(w1.shape[1]):
            up, down = w1.copy(), w1.copy()
            up[i, j] += h
            down[i, j] -= h
            numeric[i, j] = (loss_of(up, b1, w2, b2) -
                             loss_of(down, b1, w2, b2)) / (2 * h)
    err = float(np.abs(grads['w1'] - numeric).max())
    assert err < 1e-6, f'dL/dW1 differs by {err:.2e}'
    return f'hand-derived gradients match numeric to {err:.1e}'


def _g_dimensionality(impl) -> str:
    for d, eps in ((2, 0.05), (10, 0.05), (100, 0.05)):
        got = impl.shell_fraction(d, eps)
        want = 1.0 - (1.0 - eps) ** d
        assert abs(got - want) < 1e-12, f'd={d}'
    high = impl.shell_fraction(100, 0.05)
    assert high > 0.99, 'at d=100 almost all volume is in the shell'
    return f'at d=100, {high:.3%} of the volume is within 5% of the surface'


def _g_entropy(impl) -> str:
    n = 8
    uniform = np.full(n, 1.0 / n)
    assert abs(impl.entropy(uniform) - np.log(n)) < 1e-12
    spike = np.zeros(n)
    spike[0] = 1.0
    assert abs(impl.entropy(spike)) < 1e-12
    return f'uniform = log {n}, point mass = 0'


def _g_kl(impl) -> str:
    rng = _rng(16)
    p = rng.random(6)
    p /= p.sum()
    assert abs(impl.kl_divergence(p, p)) < 1e-12, 'KL(p,p) must be 0'
    asymmetric = False
    for _ in range(1000):
        a, b = rng.random(6), rng.random(6)
        a, b = a / a.sum(), b / b.sum()
        kl_ab = impl.kl_divergence(a, b)
        assert kl_ab >= -1e-12, 'KL must be non-negative'
        if abs(kl_ab - impl.kl_divergence(b, a)) > 1e-6:
            asymmetric = True
    assert asymmetric, 'KL should not be symmetric'
    return 'non-negative over 1000 pairs, zero at p=q, asymmetric'


def _g_mle(impl) -> str:
    n, mu, sigma = 100_000, 1.75, 2.5
    samples = _rng(17).normal(loc=mu, scale=sigma, size=n)
    mu_hat, var_hat = impl.mle_gaussian(samples)
    standard_error = sigma / np.sqrt(n)
    assert abs(mu_hat - mu) < 3.0 * standard_error, 'mu outside 3 SE'
    assert abs(var_hat - sigma ** 2) < 0.05 * sigma ** 2, 'variance off'
    return f'mu_hat = {mu_hat:.4f} (true {mu}), var_hat = {var_hat:.4f}'


def _g_welford(impl) -> str:
    big = np.float64(1e9) + _rng(18).normal(size=50_000)
    _, variance = impl.welford(big)
    want = float(big.var(ddof=1))
    assert abs(variance - want) < 1e-6 * want, 'Welford disagrees with numpy'
    naive = float((big ** 2).mean() - big.mean() ** 2)
    return f'welford {variance:.4f} vs numpy {want:.4f}; naive gives {naive:.4f}'


def _g_bayes(impl) -> str:
    posterior = impl.bayes_posterior(0.99, 0.99, 0.001)
    assert abs(posterior - 0.0902) < 0.002, f'got {posterior:.4f}'
    return f'P(sick | positive) = {posterior:.3f} at 0.1% prevalence'

# ---- exercise harness -------------------------------------------------
# `check('1.1')` runs that exercise's gate against whatever you have
# defined in this notebook. TODO means the function is missing or still
# raises; it is never a pass.

def _impl_namespace():
    """Exposes the notebook's own functions to the gate functions."""

    class _Namespace:
        def __getattr__(self, name):
            value = globals().get(name)
            if callable(value):
                return value
            raise NotImplementedYet(f'{name} is not defined yet')

    return _Namespace()


_GATES = {}


def check(key):
    """Runs one exercise's gate and prints the verdict."""
    if key not in _GATES:
        raise KeyError(f'no exercise {key!r} in this notebook')
    title, gate, needs_torch = _GATES[key]
    if needs_torch and not _HAS_TORCH:
        print(f'[SKIP] {key}  {title}')
        print('       PyTorch is not available in this runtime.')
        return
    try:
        detail = gate(_impl_namespace())
        print(f'[PASS] {key}  {title}')
        if detail:
            print(f'       {detail}')
    except NotImplementedYet as todo:
        print(f'[TODO] {key}  {title}')
        print(f'       not implemented yet: {todo}')
    except AssertionError as failure:
        print(f'[FAIL] {key}  {title}')
        print(f'       {failure or "gate failed"}')
    except Exception as error:                      # noqa: BLE001
        print(f'[FAIL] {key}  {title}')
        print(f'       {type(error).__name__}: {error}')


def check_all():
    """Runs every gate in this notebook and prints a summary."""
    counts = {'PASS': 0, 'FAIL': 0, 'TODO': 0, 'SKIP': 0}
    for key, (title, gate, needs_torch) in _GATES.items():
        if needs_torch and not _HAS_TORCH:
            counts['SKIP'] += 1
            continue
        try:
            gate(_impl_namespace())
            counts['PASS'] += 1
        except NotImplementedYet:
            counts['TODO'] += 1
        except Exception:                           # noqa: BLE001
            counts['FAIL'] += 1
    print(f'passed {counts["PASS"]}   failed {counts["FAIL"]}   '
          f'todo {counts["TODO"]}   skipped {counts["SKIP"]}')
    if counts['TODO']:
        print(f'\n{counts["TODO"]} still TODO. A TODO is not a pass.')


_GATES = {
    '1.1': ('l_p norms match NumPy', _g_norm, False),
    '1.2': ('Vectorisation speedup', _g_vectorisation, False),
    '1.3': ('Rank detects collinearity', _g_rank, False),
    '2.1': ('Gram-Schmidt orthonormality', _g_gram_schmidt, False),
    '2.2': ('Classical vs modified on a Hilbert matrix', _g_gram_schmidt_stability, False),
    '2.3': ('Projection is idempotent and symmetric', _g_projection, False),
    '3.1': ('Rank-nullity theorem', _g_rank_nullity, False),
    '3.2': ('Rotations preserve length', _g_rotation, False),
    '4.1': ('LU decomposition: PA = LU', _g_lu, False),
    '4.2': ('Solving from the factorisation', _g_lu_solve, False),
    '4.3': ('Determinant via LU', _g_det, False),
    '4.4': ('Solve beats explicit inversion', _g_conditioning, False),
    '5.1': ('Power iteration', _g_power_iteration, False),
    '5.2': ('Eckart-Young low-rank error', _g_eckart_young, False),
    '5.3': ('PCA from the SVD', _g_pca, False),
    '5.4': ('Eigenvalues of a 2x2 by hand', _g_eig_2x2, False),
    '5.5': ('Singular values from AtA', _g_singular_values, False),
    '6.1': ('The U-shaped differentiation error', _g_central_difference, False),
    '6.2': ('Gradient descent converges below 2/L', _g_gd_converges, False),
    '6.3': ('Gradient descent diverges above 2/L', _g_gd_diverges, False),
    '7.1': ('Numeric Jacobian', _g_jacobian, False),
    '7.2': ('Backpropagation by hand', _g_backprop, False),
    '7.3': ('Curse of dimensionality', _g_dimensionality, False),
    '8.1': ('Entropy', _g_entropy, False),
    '8.2': ('KL divergence properties', _g_kl, False),
    '8.3': ('Maximum likelihood recovers parameters', _g_mle, False),
    '8.4': ('Welford beats the naive variance', _g_welford, False),
    '8.5': ('Bayes and the base rate', _g_bayes, False),
}

print(f'setup complete: {len(_GATES)} exercises loaded, torch={_HAS_TORCH}')

## 1.1 · l_p norms match NumPy

*Read first: Ch 2 · MML p.49*

In [ ]:
def lp_norm(x: np.ndarray, p: float) -> float:
    """Returns the l_p norm of a vector. [Ch 2 · MML p.49]"""
    raise NotImplementedYet('1.1 lp_norm')

In [ ]:
check('1.1')

## 1.2 · Vectorisation speedup

*Read first: Ch 1 · MML p.37*

Nothing to implement here — this one is an **observation**. Run the
check and read the number it reports; it is about your machine or the
mathematics, not about your code.

In [ ]:
check('1.2')

## 1.3 · Rank detects collinearity

*Read first: Ch 1 · MML p.15*

Nothing to implement here — this one is an **observation**. Run the
check and read the number it reports; it is about your machine or the
mathematics, not about your code.

In [ ]:
check('1.3')

## 2.1 · Gram-Schmidt orthonormality

*Read first: Ch 2 · MML p.89*

In [ ]:
def gram_schmidt_classical(a: np.ndarray) -> np.ndarray:
    """Orthonormalises columns, classical version. [Ch 2 · MML p.67]"""
    raise NotImplementedYet('2.1 gram_schmidt_classical')

In [ ]:
check('2.1')

## 2.2 · Classical vs modified on a Hilbert matrix

*Read first: Ch 2 · MML p.67*

In [ ]:
def gram_schmidt_modified(a: np.ndarray) -> np.ndarray:
    """Orthonormalises columns, modified version. [Ch 2 · MML p.89]"""
    raise NotImplementedYet('2.2 gram_schmidt_modified')

In [ ]:
check('2.2')

## 2.3 · Projection is idempotent and symmetric

*Read first: Ch 2*

In [ ]:
def projection_matrix(v: np.ndarray) -> np.ndarray:
    """Returns the matrix projecting onto span(v). [Ch 2]"""
    raise NotImplementedYet('2.3 projection_matrix')

In [ ]:
check('2.3')

## 3.1 · Rank-nullity theorem

*Read first: Ch 3 · MML p.127*

Nothing to implement here — this one is an **observation**. Run the
check and read the number it reports; it is about your machine or the
mathematics, not about your code.

In [ ]:
check('3.1')

## 3.2 · Rotations preserve length

*Read first: Ch 3 · MML p.134*

Nothing to implement here — this one is an **observation**. Run the
check and read the number it reports; it is about your machine or the
mathematics, not about your code.

In [ ]:
check('3.2')

## 4.1 · LU decomposition: PA = LU

*Read first: Ch 4 · MML p.171*

In [ ]:
def lu_decompose(a: np.ndarray) -> Tuple[np.ndarray, np.ndarray,
                                         np.ndarray]:
    """Returns (P, L, U) with PA = LU. [Ch 4 · MML p.171]"""
    raise NotImplementedYet('4.1 lu_decompose')

In [ ]:
check('4.1')

## 4.2 · Solving from the factorisation

*Read first: Ch 4*

In [ ]:
def lu_solve(p: np.ndarray, l: np.ndarray, u: np.ndarray,
             b: np.ndarray) -> np.ndarray:
    """Solves Ax = b from a PA = LU factorisation. [Ch 4]"""
    raise NotImplementedYet('4.2 lu_solve')

In [ ]:
check('4.2')

## 4.3 · Determinant via LU

*Read first: Ch 4 · MML p.182*

In [ ]:
def det_via_lu(a: np.ndarray) -> float:
    """Returns det(A) from its LU factorisation. [Ch 4 · MML p.182]"""
    raise NotImplementedYet('4.3 det_via_lu')

In [ ]:
check('4.3')

## 4.4 · Solve beats explicit inversion

*Read first: Ch 4 · MML p.177*

Nothing to implement here — this one is an **observation**. Run the
check and read the number it reports; it is about your machine or the
mathematics, not about your code.

In [ ]:
check('4.4')

## 5.1 · Power iteration

*Read first: Ch 5 · MML p.217*

In [ ]:
def power_iteration(a: np.ndarray) -> Tuple[float, np.ndarray]:
    """Returns the dominant eigenpair. [Ch 5 · MML p.217]"""
    raise NotImplementedYet('5.1 power_iteration')

In [ ]:
check('5.1')

## 5.2 · Eckart-Young low-rank error

*Read first: Ch 5 · MML p.207*

In [ ]:
def svd_lowrank(a: np.ndarray, k: int) -> np.ndarray:
    """Returns the best rank-k approximation. [Ch 5 · MML p.207]"""
    raise NotImplementedYet('5.2 svd_lowrank')

In [ ]:
check('5.2')

## 5.3 · PCA from the SVD

*Read first: Ch 5*

In [ ]:
def pca(x: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (components, explained_variance_ratio). [Ch 5]"""
    raise NotImplementedYet('5.3 pca')

In [ ]:
check('5.3')

## 5.4 · Eigenvalues of a 2x2 by hand

*Read first: Ch 5 · MML p.192*

In [ ]:
def eigenvalues_2x2(a: np.ndarray) -> Tuple[float, float]:
    """Eigenvalues of a 2x2 via the characteristic polynomial. [Ch 5]"""
    raise NotImplementedYet('5.4 eigenvalues_2x2')

In [ ]:
check('5.4')

## 5.5 · Singular values from AtA

*Read first: Ch 5 · MML p.207*

In [ ]:
def singular_values(a: np.ndarray) -> np.ndarray:
    """Singular values as sqrt of the eigenvalues of AᵀA. [Ch 5]"""
    raise NotImplementedYet('5.5 singular_values')

In [ ]:
check('5.5')

## 6.1 · The U-shaped differentiation error

*Read first: Ch 6*

In [ ]:
def central_difference(f: Callable, x: float, h: float) -> float:
    """Approximates f'(x) by central difference. [Ch 6]"""
    raise NotImplementedYet('6.1 central_difference')

In [ ]:
check('6.1')

## 6.2 · Gradient descent converges below 2/L

*Read first: Ch 6 · MML p.381*

In [ ]:
def gradient_descent(grad: Callable, x0: np.ndarray, lr: float,
                     iters: int, momentum: float = 0.0) -> np.ndarray:
    """Minimises via gradient descent. [Ch 6 · MML p.381, p.481]"""
    raise NotImplementedYet('6.2 gradient_descent')

In [ ]:
check('6.2')

## 6.3 · Gradient descent diverges above 2/L

*Read first: Ch 6 · MML p.389*

Nothing to implement here — this one is an **observation**. Run the
check and read the number it reports; it is about your machine or the
mathematics, not about your code.

In [ ]:
check('6.3')

## 7.1 · Numeric Jacobian

*Read first: Ch 7 · MML p.460*

In [ ]:
def numeric_jacobian(f: Callable, x: np.ndarray,
                     h: float = 1e-6) -> np.ndarray:
    """Approximates the Jacobian of f at x. [Ch 7 · MML p.460]"""
    raise NotImplementedYet('7.1 numeric_jacobian')

In [ ]:
check('7.1')

## 7.2 · Backpropagation by hand

*Read first: Ch 7*

In [ ]:
def two_layer_backward(w1: np.ndarray, b1: np.ndarray, w2: np.ndarray,
                       b2: np.ndarray, x: np.ndarray,
                       y: float) -> Tuple[float, Dict[str, np.ndarray]]:
    """Forward and backward pass of a two-layer tanh net. [Ch 7]"""
    raise NotImplementedYet('7.2 two_layer_backward')

In [ ]:
check('7.2')

## 7.3 · Curse of dimensionality

*Read first: Ch 7 · MML p.443*

In [ ]:
def shell_fraction(d: int, eps: float) -> float:
    """Fraction of a unit ball's volume in the outer shell. [Ch 7]"""
    raise NotImplementedYet('7.3 shell_fraction')

In [ ]:
check('7.3')

## 8.1 · Entropy

*Read first: Ch 8 · MML p.606*

In [ ]:
def entropy(p: np.ndarray) -> float:
    """Shannon entropy in nats. [Ch 8 · MML p.606]"""
    raise NotImplementedYet('8.1 entropy')

In [ ]:
check('8.1')

## 8.2 · KL divergence properties

*Read first: Ch 8 · MML p.606*

In [ ]:
def kl_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """KL divergence from q to p, in nats. [Ch 8 · MML p.606]"""
    raise NotImplementedYet('8.2 kl_divergence')

In [ ]:
check('8.2')

## 8.3 · Maximum likelihood recovers parameters

*Read first: Ch 8 · MML p.619*

In [ ]:
def mle_gaussian(samples: np.ndarray) -> Tuple[float, float]:
    """Maximum-likelihood (mu, sigma^2). [Ch 8 · MML p.619]"""
    raise NotImplementedYet('8.3 mle_gaussian')

In [ ]:
check('8.3')

## 8.4 · Welford beats the naive variance

*Read first: Ch 8 · MML p.589*

In [ ]:
def welford(values: np.ndarray) -> Tuple[float, float]:
    """One-pass stable (mean, sample variance). [Ch 8 · MML p.589]"""
    raise NotImplementedYet('8.4 welford')

In [ ]:
check('8.4')

## 8.5 · Bayes and the base rate

*Read first: Ch 8 · MML p.515*

In [ ]:
def bayes_posterior(sensitivity: float, specificity: float,
                    prevalence: float) -> float:
    """P(disease | positive test). [Ch 8 · MML p.515]"""
    raise NotImplementedYet('8.5 bayes_posterior')

In [ ]:
check('8.5')

---

## Summary

Run this once you have attempted everything above.

In [ ]:
check_all()